In [18]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

In [19]:
# Spark Session
spark = SparkSession.builder.appName("Spark-Iceberg-Test").getOrCreate()

# Spark Context
spark.sparkContext.setLogLevel("ERROR")

print("NOTE: SparkSession created with Iceberg and S3 Tables support")

NOTE: SparkSession created with Iceberg and S3 Tables support


In [11]:
spark.sql("""
    SELECT * FROM lakehouse_prod.nyc.taxis_01
""").show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [15]:
# Define schema for NYC Taxi Data
schema = StructType([
    StructField('VendorID', LongType(), True),
    StructField('tpep_pickup_datetime', TimestampNTZType(), True),
    StructField('tpep_dropoff_datetime', TimestampNTZType(), True),
    StructField('passenger_count', DoubleType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('RatecodeID', DoubleType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('PULocationID', LongType(), True),
    StructField('DOLocationID', LongType(), True),
    StructField('payment_type', LongType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType(), True),
    StructField('airport_fee', DoubleType(), True)
])

In [ ]:
# Read Parquet file from MinIO
parquet_path = './data/yellow_tripdata_2021-04.parquet'
df = spark.read.option("header", "true").schema(schema).parquet(parquet_path)
df.show(10)

In [ ]:
df.writeTo("lakehouse_prod.nyc.taxis_01") \
  .option("location", "s3a://lakehouse-dev-bucket/external_test/taxis_01") \
  .partitionedBy("tpep_pickup_datetime") \
  .replace()

In [12]:
df = spark.read.format("iceberg").load("lakehouse_prod.nyc.taxis_01")
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [13]:
df.createOrReplaceTempView("tempViewTest")

In [14]:
spark.sql("""
    SELECT * FROM tempViewTest
""").show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [17]:
spark.stop()